In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import CIFAR10, CIFAR100, MNIST, STL10
import torchvision.transforms as T
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mhnlib.utils as mhn_utils
from math import log, sqrt
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path
import re
from einops import rearrange
import pandas as pd
import networkx as nx
import glasbey
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from sklearn.decomposition import PCA
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
#device = torch.device("cpu")

# Utils

In [ ]:
from scipy.cluster.hierarchy import (
    linkage,
    leaves_list,
    optimal_leaf_ordering,
)
from scipy.spatial.distance import pdist, squareform
def hierarchical_haar_from_similarity(S, method="average"):
    S = np.asarray(S)
    N = S.shape[0]

    # similarity -> dissimilarity
    D = S.max() - S
    np.fill_diagonal(D, 0.0)

    d = squareform(D, checks=False)

    Z = linkage(d, method=method)
    Z = optimal_leaf_ordering(Z, d)

    clusters = {i: np.array([i], dtype=int) for i in range(N)}
    wavelets = []

    active = list(range(N))

    # scaling vectors representing the active clusters at each split
    hierarchy = [np.eye(N)]

    for k, row in enumerate(Z):
        a = int(row[0])
        b = int(row[1])

        A = clusters[a]
        B = clusters[b]

        nA = len(A)
        nB = len(B)
        n = nA + nB

        # Haar detail mode
        h = np.zeros(N)
        h[A] = np.sqrt(nB / (nA * n))
        h[B] = -np.sqrt(nA / (nB * n))

        wavelets.append(h)

        # new cluster
        new_node = N + k
        clusters[new_node] = np.concatenate([A, B])

        # update active nodes
        active.remove(a)
        active.remove(b)
        active.append(new_node)

        # scaling vectors for all current clusters
        P = np.zeros((len(active), N))

        for j, node in enumerate(active):
            inds = clusters[node]
            P[j, inds] = 1.0 / len(inds)

        hierarchy.append(P)

    scaling = np.ones(N) / np.sqrt(N)

    H = np.vstack([scaling] + wavelets[::-1])

    return H, Z, clusters, leaves_list(Z), hierarchy
def hierarchical_haar(X, method="average", metric="euclidean"):
    X = np.asarray(X)
    N = X.shape[0]

    distances = pdist(X, metric=metric)

    Z = linkage(distances, method=method)
    Z = optimal_leaf_ordering(Z, distances)

    clusters = {i: np.array([i], dtype=int) for i in range(N)}
    wavelets = []

    # active tree nodes
    active = list(range(N))

    # coarse objects at every hierarchy
    hierarchy = [X.copy()]

    for k, row in enumerate(Z):
        a = int(row[0])
        b = int(row[1])

        A = clusters[a]
        B = clusters[b]

        nA = len(A)
        nB = len(B)
        n = nA + nB

        # Haar detail
        h = np.zeros(N)
        h[A] = np.sqrt(nB / (nA * n))
        h[B] = -np.sqrt(nA / (nB * n))

        wavelets.append(h)

        # new cluster
        new_node = N + k
        clusters[new_node] = np.concatenate([A, B])

        # update active clusters
        active.remove(a)
        active.remove(b)
        active.append(new_node)

        # coarse object for every active cluster
        X_level = np.stack([
            X[clusters[node]].sum(axis=0)
            / len(clusters[node])
            for node in active
        ])

        hierarchy.append(X_level)

    scaling = np.ones(N) / np.sqrt(N)

    # coarse -> fine
    H = np.vstack([scaling] + wavelets[::-1])

    return H, Z, clusters, leaves_list(Z), hierarchy

In [ ]:
from numba import njit
@njit(nogil=True)
def distance_tree(data_list, threshold : float):
    edges_from = []
    edges_to = []
    num_levels = len(data_list)
    for level in range(0, num_levels-1):
        num_elements_level = len(data_list[level])
        num_elements_next_level = len(data_list[level+1])
        
        for idx in range(0, num_elements_level):
            sq_norm_i = np.sum(data_list[level][idx]**2)
            num_children_found = 0
            for jdx in range(0, num_elements_next_level):
                sq_norm_j = np.sum(data_list[level+1][jdx]**2)
                dist_ij = sq_norm_i + sq_norm_j - 2*np.sum(data_list[level][idx]*data_list[level+1][jdx])
                if dist_ij <= threshold:
                    edges_from.append((level, idx))
                    edges_to.append((level+1, jdx))
                    num_children_found += 1
    return np.array(edges_from), np.array(edges_to)

def fisher_distance(a, b, w):
    diff = a - b
    valid = w > 0

    terms = torch.where(
        valid,
        diff.square() / torch.where(valid, w, 1.0),
        0.0,
    )

    return terms.sum(dim=-1).sqrt()

def disambiguate_fixed_points(x_fp_list, weights, threshold_or_thresholds, complete_linkage):
    if isinstance(threshold_or_thresholds, float):
        thresholds = [threshold_or_thresholds] * len(x_fp_list)
    else:
        thresholds = threshold_or_thresholds
    pbar = tqdm(range(len(x_fp_list)))
    num_fps = [ ]
    x_fps = []
    w_fps = []
    for idx in pbar:
        x_unique, x_unique_counts, labels = mhn_utils.group_by_distance(x_fp_list[idx], float(thresholds[idx]), complete_linkage)
        w_unique = torch.zeros(x_unique.shape[0], weights.shape[-1])
        w_unique.index_add_(0, labels, weights[idx])
        w_unique /= x_unique_counts[:, None]
        num_fps.append(x_unique.shape[0])
        x_fps.append(torch.as_tensor(x_unique))
        w_fps.append(w_unique)
        pbar.set_postfix({"Found clusters" : num_fps[-1] })
    num_fps = torch.as_tensor(num_fps)
    return x_fps, w_fps, num_fps

#def disambiguate_fixed_points(x_fp_list, weights, threshold, non_decreasing, complete_linkage):
#    pbar = tqdm(range(len(x_fp_list)))
#    num_fps = [ ]
#    x_fps = []
#    w_fps = []
#    for idx in pbar:
#        x_unique, x_unique_counts, labels = mhn_utils.group_by_distance(x_fp_list[idx], threshold, complete_linkage)
#        w_unique = torch.zeros(x_unique.shape[0], weights.shape[-1])
#        w_unique.index_add_(0, labels, weights[idx])
#        w_unique /= x_unique_counts[:, None]
#        num_fps.append(x_unique.shape[0])
#        x_fps.append(torch.as_tensor(x_unique))
#        w_fps.append(w_unique)
#        if len(num_fps) > 1 and non_decreasing:
#            if num_fps[-1] < num_fps[-2]:
#                raise Exception("No valid cluster possible.")
#        pbar.set_postfix({"Found clusters" : num_fps[-1] })
#    num_fps = torch.as_tensor(num_fps)
#    return x_fps, w_fps, num_fps
def disambiguate_fixed_points_by_similarity(x_fp_list, weights, threshold_or_thresholds, non_decreasing, complete_linkage):
    if isinstance(threshold_or_thresholds, float):
        thresholds = [threshold_or_thresholds] * len(x_fp_list)
    else:
        thresholds = threshold_or_thresholds
    pbar = tqdm(range(len(x_fp_list)))
    num_fps = [ ]
    x_fps = []
    w_fps = []
    for idx in pbar:
        x_unique, x_unique_counts, labels = mhn_utils.group_by_similarity(x_fp_list[idx], threshold, complete_linkage)
        w_unique = torch.zeros(x_unique.shape[0], weights.shape[-1])
        w_unique.index_add_(0, labels, weights[idx])
        w_unique /= x_unique_counts[:, None]
        num_fps.append(x_unique.shape[0])
        x_fps.append(torch.as_tensor(x_unique))
        w_fps.append(w_unique)
        if len(num_fps) > 1 and non_decreasing:
            if num_fps[-1] < num_fps[-2]:
                raise Exception("No valid cluster possible.")
        pbar.set_postfix({"Found clusters" : num_fps[-1] })
    num_fps = torch.as_tensor(num_fps)
    return x_fps, w_fps, num_fps
def disambiguate_fixed_points_by_jensen(w_fp_list, x_fp_list, threshold_or_thresholds, non_decreasing, complete_linkage):
    if isinstance(threshold_or_thresholds, float):
        thresholds = [threshold_or_thresholds] * len(x_fp_list)
    else:
        thresholds = threshold_or_thresholds
    pbar = tqdm(range(len(x_fp_list)))
    num_fps = [ ]
    x_fps = []
    w_fps = []
    for idx in pbar:
        w_unique, w_unique_counts, labels = mhn_utils.group_by_jensen(w_fp_list[idx], threshold, complete_linkage)
        x_unique = torch.zeros(w_unique.shape[0], x_fp_list[idx].shape[-1])
        x_unique.index_add_(0, labels, x_fp_list[idx])
        x_unique /= w_unique_counts[:, None]
        num_fps.append(w_unique.shape[0])
        x_fps.append(torch.as_tensor(x_unique))
        w_fps.append(w_unique)
        if len(num_fps) > 1 and non_decreasing:
            if num_fps[-1] < num_fps[-2]:
                raise Exception("No valid cluster possible.")
        pbar.set_postfix({"Found clusters" : num_fps[-1] })
    num_fps = torch.as_tensor(num_fps)
    return x_fps, w_fps, num_fps


# Select dataset type

In [ ]:
DATASET = "mnist"
IS_IMAGE = DATASET in ["mnist", "stl10", "cifar10", "cifar100"]
data_path = Path("paper_results/experiments/")
quenched_files = list(data_path.glob(f"{DATASET}*weights_quenched*.pt"))
annealed_files = list(data_path.glob(f"{DATASET}*weights_annealed*.pt"))
file_df = {'is_quenched' : [], 'num_per_label' : [], 'noise' : [], 'is_euclidean' : [], 'is_centered' : [], 'file' : []}
for file in quenched_files + annealed_files:
    is_quenched = 'quenched' in file.stem
    num_per_label = re.search(r'per_label=(\d+)', file.stem)
    if num_per_label:
        num_per_label = int(num_per_label.group(1))
    else:
        num_per_label = np.nan
    is_euclidean = 'euclidean=True' in file.stem
    is_centered = 'centered=True' in file.stem
    logit_noise = re.search(r'noise=(\d+\.?\d*)', file.stem)
    if logit_noise:
        logit_noise = float(logit_noise.group(1))
    else:
        logit_noise = None
    file_df['is_quenched'].append(is_quenched)
    file_df['num_per_label'].append(num_per_label)
    file_df['noise'].append(logit_noise)
    file_df['is_euclidean'].append(is_euclidean)
    file_df['is_centered'].append(is_centered)
    file_df['file'].append(str(file.absolute()))
file_df = pd.DataFrame.from_dict(file_df)

# Single experiment analysis

In [ ]:
is_centered = True
is_euclidean = False
sub_file_df = file_df[(file_df['is_euclidean'] == is_euclidean) & (file_df['is_centered'] == is_centered)]
logit_noise = 2.0
num_per_label = 1 if IS_IMAGE else None
if num_per_label is not None:
    quenched_file_mask = (file_df['is_euclidean'] == is_euclidean) & (file_df['is_centered'] == is_centered) & (file_df['is_quenched'] == True) & (file_df['num_per_label'] == num_per_label)
    if logit_noise is not None:
        quenched_file_mask = quenched_file_mask & (file_df['noise'] == logit_noise)
    else:
        quenched_file_mask = quenched_file_mask & pd.isna(file_df['noise'])
    annealed_file_mask = (file_df['is_euclidean'] == is_euclidean) & (file_df['is_centered'] == is_centered) & (file_df['is_quenched'] == False) & (file_df['num_per_label'] == num_per_label) & (file_df['noise'] == logit_noise)
else:
    quenched_file_mask = (file_df['is_euclidean'] == is_euclidean) & (file_df['is_centered'] == is_centered) & (file_df['is_quenched'] == True) 
    if logit_noise is not None:
        quenched_file_mask = quenched_file_mask & (file_df['noise'] == logit_noise)
    else:
        quenched_file_mask = quenched_file_mask & pd.isna(file_df['noise'])
    annealed_file_mask = (file_df['is_euclidean'] == is_euclidean) & (file_df['is_centered'] == is_centered) & (file_df['is_quenched'] == False)  & (file_df['noise'] == logit_noise)

In [ ]:
quenched_file_name = file_df[quenched_file_mask]['file'].iloc[0]
quenched_data = torch.load(quenched_file_name)
quenched_weights = quenched_data['weights']
shift = quenched_data['shift']
rms = quenched_data['rms']
betas = quenched_data['betas']
biases = quenched_data['biases']
patterns = quenched_data['patterns']
gram = patterns @ patterns.T
quenched_x = quenched_weights @ patterns

In [ ]:
#annealed_file_name = file_df[annealed_file_mask]['file'].iloc[0]
#annealed_data = torch.load(annealed_file_name)
#annealed_weights = annealed_data['weights']
#annealed_x = annealed_weights @ patterns

In [ ]:
data_dict = torch.load(Path(quenched_file_name).parent / (f"{DATASET}_per_label={num_per_label}.pt" if IS_IMAGE else f"{DATASET}.pt"))
labels = data_dict['labels']
label_to_name = data_dict['label_to_name'] if 'label_to_name' in data_dict else None
data = data_dict['data']

## Derived quantities

In [ ]:
#if not is_euclidean:
#    quenched_thresholds =  1.0 - torch.linspace(0.0, 1.0, 10)
#    for quenched_threshold in quenched_thresholds:
#        try:
#            x_quenched_fps, w_quenched_fps, num_quenched_fps = disambiguate_fixed_points_by_similarity(quenched_x, quenched_weights, quenched_threshold.item(), True, complete_linkage=True)
#            print(f"Proper clusters at {quenched_threshold}")
#            break
#        except:
#            print(f"Threshold too large {quenched_threshold}")
#else:
x_quenched_fps, w_quenched_fps, num_quenched_fps = disambiguate_fixed_points_local(quenched_x, quenched_weights, 1/torch.sqrt(betas), complete_linkage=False)


num_labels = len(torch.unique(labels))
w_quenched_per_label_fps = []
for w_fp in w_quenched_fps:
    w_per_label = torch.zeros(w_fp.shape[0], num_labels)
    w_per_label.index_add_(1, labels, w_fp)
    w_quenched_per_label_fps.append(w_per_label)

jump_indices = torch.argwhere(torch.diff(num_quenched_fps) > 0).flatten()+1
jump_indices = torch.cat([torch.zeros(1, dtype=jump_indices.dtype), jump_indices])

In [ ]:
fp_nodes = []
for beta_idx in range(len(x_quenched_fps)):
    for idx in range(len(x_quenched_fps[beta_idx])):
        fp_nodes.append((beta_idx, idx))
cc_threshold = 1e-2
edgelist_from, edgelist_to = distance_tree([x.numpy() for x in x_quenched_fps], cc_threshold)
G = nx.Graph()
G.add_nodes_from(fp_nodes)
G.add_edges_from(
    zip(map(tuple, edgelist_from), map(tuple, edgelist_to))
)
fp_connected_components = list(nx.connected_components(G))
num_fp_cc = len(fp_connected_components)
fp_tags = {}
for cc_index, cc_ in enumerate(fp_connected_components):
    cc = np.array(list(cc_))
    cc = cc[np.argsort(cc[:,0])]
    for el in cc:
        fp_tags[(el[0].item(),el[1].item())] = cc_index
fp_tag_colors = np.array([
    mcolors.to_rgb(c)
    for c in glasbey.create_palette(palette_size=num_fp_cc)
])

## Entropy (quenched)

In [ ]:
from scipy.spatial import KDTree
stab_matr = mhn_utils.get_dual_symmetric_stability_matrix(patterns @ patterns.T, torch.ones(patterns.shape[0])/patterns.shape[0], False)
beta_ev = 1.0 / torch.linalg.eigvalsh(torch.cov(patterns.T))[-1]
beta_c = 1.0 / torch.linalg.eigvalsh(stab_matr)[-1]
beta_sigma = 1.0 / torch.var(patterns, dim=0).mean()
beta_max = 1.0 / torch.norm(patterns - patterns.mean(dim=0, keepdim=True), dim=1).max()**2
beta_nn = KDTree(patterns).query(patterns, k=2)[0][:, 1].mean()**-2
fig, ax = plt.subplots(figsize=(6, 4))
quenched_entropy = mhn_utils.get_entropy(quenched_weights).mean(dim=1)
ax.plot(betas, quenched_entropy/log(patterns.shape[0]))    
ax.scatter(betas[jump_indices], quenched_entropy[ jump_indices]/log(patterns.shape[0]), color='red', s=20, label='plotted', marker='x')
ax.axvline(x=beta_c, color='black', linestyle='--', label='$\\beta_c$')
if is_euclidean:
    ax.axvline(x=beta_sigma, color='red', linestyle='--', label='$\\beta_\\sigma$')
    ax.axvline(x=beta_max, color='blue', linestyle='--', label='$\\beta_{\\rm{max \\, norm}}$')
    ax.axvline(x=beta_nn, color='green', linestyle='--', label='$\\beta_{\\rm{nn}}$')
#ax.axvline(x=beta_ev, color='orange', linestyle='--', label='$\\beta_{ev}$')
plt.legend()
ax.set_xlabel('$\\beta$')
ax.set_ylabel('$H(w)/\\log{K}$')
plt.xscale('log')
plt.savefig(f"paper_results/plots/{DATASET}_weights_quenched_num_per_label={num_per_label}_centered={is_centered}_euclidean={is_euclidean}_logit_noise={logit_noise}_entropy.pdf", bbox_inches='tight')
plt.show()

In [ ]:
if IS_IMAGE:
    from diffusers import AutoencoderKL
    ae_model = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse")
    ae_model = ae_model.to(device).eval()
    ae_model.requires_grad_(False)
    ae_scaling = ae_model.config.scaling_factor
    
    source_data_file = Path(quenched_file_name).parent
    if num_per_label is not None or np.isfinite(num_per_label):
        source_data_file = source_data_file / f"{DATASET}_per_label={num_per_label}.pt"
    else:
        source_data_file = source_data_file / f"{DATASET}.pt"
    source_data = torch.load(source_data_file)
    source_latents = source_data['latents']
    C_latents, H_latents, W_latents = source_latents.shape[1:]

    image_quenched_fps = []
    for x in x_quenched_fps:
        torch.cuda.empty_cache()
        images = ae_model.decode((x*rms + shift).view(-1,C_latents, H_latents, W_latents).to(device)).sample.cpu()
        image_quenched_fps.append(images)

    #image_annealed_fps = []
    #for x in x_annealed_fps:
    #    torch.cuda.empty_cache()
    #    images = ae_model.decode(x.view(-1,C_latents, H_latents, W_latents).to(device)).sample.cpu()
    #    image_annealed_fps.append(images)
else:
    image_quenched_fps = None
    #image_annealed_fps = None

## Class weights (quenched)

In [ ]:
class_weights_save_dir = Path(f"paper_results/plots/{DATASET}/class_weights_num_per_label={num_per_label}_centered={is_centered}_euclidean={is_euclidean}_logit_noise={logit_noise}/")
class_weights_save_dir.mkdir(parents=True, exist_ok=True)
for file in class_weights_save_dir.glob("*.pdf"):
    file.unlink()
for beta_idx in jump_indices:
    beta = betas[beta_idx]
    fig, ax = plt.subplots(figsize=(num_labels/4, 4))
    plt.imshow(w_quenched_per_label_fps[beta_idx], aspect='auto', cmap='coolwarm', vmin=0, vmax=1)
    ax.set_yticks([])
    #ax.set_yticks(np.arange(0, w_quenched_per_label_fps[beta_idx].shape[0], 1))
    ax.set_xticks(np.arange(0, w_quenched_per_label_fps[beta_idx].shape[1], 1))
    if label_to_name is not None and len(label_to_name) > 0:
        ax.set_xticklabels([label_to_name[i] for i in range(w_quenched_per_label_fps[beta_idx].shape[1])], rotation=90)
    ax.set_xlabel("Class label")
    ax.set_ylabel("Fixed point index")
    plt.savefig(class_weights_save_dir / f"beta_idx={beta_idx}.pdf", dpi=300, bbox_inches='tight')
    plt.close()

## Hierarchical clustering

In [ ]:
H, Z, clusters, order, hierarchy = hierarchical_haar(patterns)
H = torch.as_tensor(H, dtype=torch.float32)

In [ ]:
num_cols = 10
num_rows = int(np.ceil(len(hierarchy)/num_cols))
fig, axs = plt.subplots(num_rows, num_cols, figsize=(num_cols*4, num_rows*4))
if num_rows == 1:
    axs = axs[None,...]
for reverse_hierarchy_level in range(len(hierarchy)):
    hierarchy_level = len(hierarchy) - 1 - reverse_hierarchy_level
    row_idx = hierarchy_level // num_cols
    col_idx = hierarchy_level % num_cols
    x_hl = torch.as_tensor(hierarchy[hierarchy_level], dtype=torch.float32)
    x_hl_propagated = mhn_utils.deterministic_dynamics(patterns.to('cuda'), biases.to('cuda'), betas.to('cuda'), x_hl.to('cuda'), num_iterations=1000).cpu().squeeze(1)
    distances = torch.norm(x_hl_propagated - x_hl[None,...], dim=-1)
    axs[row_idx, col_idx].imshow(distances.clamp(min=1e-5).log10(), aspect='auto', cmap='seismic', vmin=-5, vmax=0)
    axs[row_idx, col_idx].set_xticks(np.arange(0, distances.shape[1], 1))
plt.show()

In [ ]:
num_cols = 10
num_rows = int(np.ceil(len(hierarchy)/num_cols))
fig, axs = plt.subplots(num_rows, num_cols, figsize=(num_cols*4, num_rows*4))
if num_rows == 1:
    axs = axs[None,...]
for hierarchy_level in range(len(hierarchy)-1, -1, -1):
    plot_idx = len(hierarchy) - 1 - hierarchy_level
    row_idx = plot_idx // num_cols
    col_idx = plot_idx % num_cols
    x_hl = torch.as_tensor(hierarchy[hierarchy_level], dtype=torch.float32)
    A =  patterns @ torch.linalg.pinv(x_hl)
    patterns_fitted = A @ x_hl
    axs[row_idx, col_idx].imshow(torch.cov(patterns_fitted.T), aspect='auto', cmap='coolwarm', vmin=-1/patterns_fitted.shape[1], vmax=1/patterns_fitted.shape[1])
plt.show()

In [ ]:
num_cols = 10
num_rows = int(np.ceil(len(jump_indices)/num_cols))
fig, axs = plt.subplots(num_rows, num_cols, figsize=(num_cols*4, num_rows*4))
if num_rows == 1:
    axs = axs[None,...]
for idx, beta_idx in enumerate(jump_indices):
    row_idx = idx // num_cols
    col_idx = idx % num_cols
    A =  patterns @ torch.linalg.pinv(x_quenched_fps[beta_idx])
    patterns_fitted = A @ x_quenched_fps[beta_idx]
    axs[row_idx, col_idx].imshow(torch.cov(patterns_fitted.T), aspect='auto', cmap='coolwarm', vmin=-1/patterns_fitted.shape[1], vmax=1/patterns_fitted.shape[1])
plt.show()

## PCA plots (quenched)

In [ ]:
pca_patterns = PCA(n_components=5).fit(patterns)
patterns_proj = pca_patterns.transform(patterns)
grayscale_conversion = torch.tensor([0.299, 0.587, 0.114])
sequences_save_dir = Path(f"paper_results/plots/{DATASET}/sequences_num_per_label={num_per_label}_centered={is_centered}_euclidean={is_euclidean}_logit_noise={logit_noise}/")
sequences_save_dir.mkdir(parents=True, exist_ok=True)
for file in sequences_save_dir.glob("*.pdf"):
    file.unlink()
for beta_idx in jump_indices:
    x_fp = x_quenched_fps[beta_idx]
    x_fp_proj = torch.as_tensor(pca_patterns.transform(x_fp))
    closest_fp_idx = ((patterns[:,None,:] - x_fp[None,:,:])**2).sum(dim=-1).argmin(dim=1)
    tags_fp = torch.as_tensor([ fp_tags[(beta_idx.item(),idx)] for idx in range(len(x_fp))]) 
    tags_patterns = torch.as_tensor([ fp_tags[(beta_idx.item(),idx.item())] for idx in closest_fp_idx])
    fig, axs = plt.subplots(ncols=2, figsize=(8,3))
    axs[0].scatter(patterns_proj[:,0], patterns_proj[:,1],color=fp_tag_colors[tags_patterns], alpha=0.3)
    axs[0].scatter(x_fp_proj[:,0], x_fp_proj[:,1], color=fp_tag_colors[tags_fp], marker='s', s=50, edgecolors='black')
    axs[0].set_xlabel("PCA #1")
    axs[0].set_ylabel("PCA #2")
    for fp_idx in range(len(x_fp_proj)):
        axs[0].text(x_fp_proj[fp_idx,0], x_fp_proj[fp_idx,1], f"{tags_fp[fp_idx]}", fontsize=15 )
    axs[1].scatter(patterns_proj[:,2], patterns_proj[:,3],color=fp_tag_colors[tags_patterns], alpha=0.3)
    axs[1].scatter(x_fp_proj[:,2], x_fp_proj[:,3], color=fp_tag_colors[tags_fp], marker='s', s=50, edgecolors='black')
    for fp_idx in range(len(x_fp_proj)):
        axs[1].text(x_fp_proj[fp_idx,2], x_fp_proj[fp_idx,3], f"{tags_fp[fp_idx]}", fontsize=15 )
    axs[1].set_xlabel("PCA #3")
    axs[1].set_ylabel("PCA #4")
    axs[0].set_aspect('equal')
    axs[1].set_aspect('equal')
    fig.suptitle(f"$\\beta={betas[beta_idx]:.2f}$")
    plt.savefig(sequences_save_dir / f"pca_quenched_beta_idx={beta_idx}.pdf", bbox_inches='tight')
    plt.close()
    if IS_IMAGE:
        images = image_quenched_fps[beta_idx]
        num_images = images.shape[0]
        fig, axs = plt.subplots(ncols=images.shape[0], figsize=(4*images.shape[0],4))
        if images.shape[0] == 1:
            axs = [axs]
        for idx in range(images.shape[0]):
            if DATASET == "mnist":
                axs[idx].imshow((1+(images[idx].permute(1,2,0) @ grayscale_conversion).clip(-1,1))/2, vmin=0, vmax=1, cmap='gray')
            else:
                axs[idx].imshow((1+images[idx].permute(1,2,0).clip(-1,1))/2, vmin=0, vmax=1, cmap='gray')
            axs[idx].set_title(fp_tags[(beta_idx.item(),idx)])
        plt.savefig(sequences_save_dir / f"images_quenched_beta_idx={beta_idx}.pdf", bbox_inches='tight')
        plt.close()

## Matrices (quenched)

In [ ]:
from sklearn.cluster import AgglomerativeClustering
import seaborn as sns
gram_min = torch.min(gram)
gram_max = torch.max(gram)
dist_matrix = torch.cdist(patterns, patterns, p=2)
dist_min = torch.min(dist_matrix)
dist_max = torch.max(dist_matrix)
#cov_patterns = torch.cov(patterns.T)
#cov_vals = torch.linalg.eigvalsh(cov_patterns)
#hist_cum, edges_cum = torch.histogram(cov_vals, bins=100, density=True)
#hist_cum = torch.cumsum(hist_cum*torch.diff(edges_cum), dim=0)
matrices_save_dir = Path(f"paper_results/plots/{DATASET}/matrices_num_per_label={num_per_label}_centered={is_centered}_euclidean={is_euclidean}_logit_noise={logit_noise}/")
matrices_save_dir.mkdir(parents=True, exist_ok=True)
for file in matrices_save_dir.glob("*.pdf"):
    file.unlink()
for beta_idx in jump_indices:
    x_fp = x_quenched_fps[beta_idx]
    A =  patterns @ torch.linalg.pinv(x_fp)
    patterns_fitted = A @ x_fp
    gram_fitted = patterns_fitted @ patterns_fitted.T
    dist_matrix_fitted = torch.cdist(patterns_fitted, patterns_fitted, p=2)
    
    
    #if len(x_fp) > 1:
    #    cov_fp = torch.cov(x_fp.T)
    #    cov_vals_fp = torch.linalg.eigvalsh(cov_fp)
    #    hist_fp_cum, edges_fp_cum = torch.histogram(cov_vals_fp, bins=100, density=True)
    #    hist_fp_cum = torch.cumsum(hist_fp_cum*torch.diff(edges_fp_cum), dim=0)
    #    plt.plot(edges_cum[:-1].numpy(), hist_cum.numpy(), label='Patterns', color='blue')
    #    plt.plot(edges_fp_cum[:-1].numpy(), hist_fp_cum.numpy(), label='Fixed Points', color='orange')
    #    #plt.plot(np.sort(cov_vals.numpy()), np.linspace(0, 1, len(cov_vals)), label='Patterns', color='blue')
    #    #plt.plot(np.sort(cov_vals_fp.numpy()), np.linspace(0, 1, len(cov_vals_fp)), label='Fixed Points', color='orange')
    #    plt.xlabel("Eigenvalue")
    #    plt.ylabel("Count")
    #    plt.title(f"Covariance Eigenvalue Distribution at $\\beta={betas[beta_idx]:.2f}$")
    #    plt.legend()
    #    plt.show()
    fig, axs = plt.subplots(ncols=5, figsize=(30,6))
    axs[0].imshow(gram_fitted, vmin=gram_min, vmax=gram_max, cmap='bwr')
    axs[0].set_xlabel("Pattern Index")
    axs[0].set_ylabel("Pattern Index")
    axs[0].set_title("Fitted Patterns Gram Matrix")
    axs[1].imshow(gram, vmin=gram_min, vmax=gram_max, cmap='bwr')
    axs[1].set_xlabel("Pattern Index")
    axs[1].set_ylabel("Pattern Index")
    axs[1].set_title("Patterns Gram Matrix")
    if label_to_name is not None and len(label_to_name) > 0:
        axs[0].set_xticks(np.arange(0, gram_fitted.shape[0], 1))
        axs[0].set_xticklabels([label_to_name[i] for i in range(gram_fitted.shape[0])], rotation=90, fontsize=8)
        axs[0].set_yticks(np.arange(0, gram_fitted.shape[0], 1))
        axs[0].set_yticklabels([label_to_name[i] for i in range(gram_fitted.shape[0])], fontsize=8)
        axs[1].set_xticks(np.arange(0, gram.shape[0], 1))
        axs[1].set_xticklabels([label_to_name[i] for i in range(gram.shape[0])], rotation=90, fontsize=8)
        axs[1].set_yticks(np.arange(0, gram.shape[0], 1))
        axs[1].set_yticklabels([label_to_name[i] for i in range(gram.shape[0])], fontsize=8)
    axs[2].imshow(x_fp @ x_fp.T, vmin=gram_min, vmax=gram_max, cmap='bwr')
    axs[2].set_xlabel("Fixed Point Index")
    axs[2].set_ylabel("Fixed Point Index")
    axs[2].set_title("Fixed Points Gram Matrix")
    axs[3].imshow(dist_matrix_fitted, vmin=dist_min, vmax=dist_max, cmap='viridis')
    axs[3].set_xlabel("Pattern Index")
    axs[3].set_ylabel("Pattern Index")
    axs[3].set_title("Fitted Patterns Distance Matrix")
    axs[4].imshow(dist_matrix, vmin=dist_min, vmax=dist_max, cmap='viridis')
    axs[4].set_xlabel("Pattern Index")
    axs[4].set_ylabel("Pattern Index")
    axs[4].set_title("Patterns Distance Matrix")
    for ax in axs:
        fig.colorbar(ax.images[0], ax=ax, use_gridspec=True, location='right', shrink=0.8)
    plt.savefig(matrices_save_dir / f"beta_idx={beta_idx}.pdf", bbox_inches='tight')
    plt.close()

# Multi-noise analysis

In [ ]:
num_per_label = 1 if IS_IMAGE else None
is_euclidean = True
is_centered = False
sub_file_df = file_df[(file_df['is_euclidean'] == is_euclidean) & (file_df['is_centered'] == is_centered)  & ~pd.isna(file_df['noise'])]
if num_per_label is not None:
    sub_file_df = sub_file_df[sub_file_df['num_per_label'] == num_per_label]
sub_file_df = sub_file_df.sort_values(by=['noise'])

In [ ]:
database = {}
for _,file in sub_file_df.iterrows():
    quenched_data = torch.load(file['file'])
    quenched_weights = quenched_data['weights']
    shift = quenched_data['shift']
    rms = quenched_data['rms']
    betas = quenched_data['betas']
    biases = quenched_data['biases']
    patterns = quenched_data['patterns']
    gram = patterns @ patterns.T
    quenched_x = quenched_weights @ patterns
    mean_entropy = mhn_utils.get_entropy(quenched_weights).mean(dim=1)
    database[file.noise] = {"patterns" : patterns,
                             "quenched_weights" : quenched_weights,
                               "quenched_x" : quenched_x,
                                 "mean_entropy" : mean_entropy,
                                  "betas" : betas, "shift" : shift, "rms" : rms, "biases" : biases}

In [ ]:
noise_colors = { noise : cm.coolwarm(noise / max(database.keys())) for noise in database.keys() }

In [ ]:
for noise, data in database.items():
    betas = data['betas']
    mean_entropy = data['mean_entropy']
    plt.plot(betas, mean_entropy, label=f"noise={noise}", color=noise_colors[noise], marker='.')
plt.xscale('log')
plt.show()

In [ ]:
for noise, data in database.items():

    betas = data["betas"]

    K = data["patterns"].shape[0]

    mean_entropy = data["mean_entropy"]
    scaled_mean_entropy = mean_entropy / log(K)

    uniform_idx = torch.argwhere(
        scaled_mean_entropy > 0.999
    ).flatten()

    if len(uniform_idx) == 0:
        start_idx = 0

    start_idx = uniform_idx[-1].item()

    x_quenched_fps, w_quenched_fps, num_quenched_fps = (
        disambiguate_fixed_points(
            data["quenched_x"][start_idx:],
            data["quenched_weights"][start_idx:],
            data["betas"][start_idx:].rsqrt(),
            complete_linkage=False,
        )
    )

    # Before start_idx, force the unique uniform/high-entropy branch
    x_quenched_fps = (
        list(data["quenched_x"][:start_idx].mean(dim=1))
        + x_quenched_fps
    )

    w_quenched_fps = (
        list(data["quenched_weights"][:start_idx].mean(dim=1))
        + w_quenched_fps
    )

    num_quenched_fps = torch.cat([
        torch.ones(start_idx, dtype=torch.long),
        num_quenched_fps,
    ])

    jump_indices = (
        torch.argwhere(
            torch.diff(num_quenched_fps) > 0
        ).flatten()
        + 1
    )

    jump_indices = torch.cat([
        torch.zeros(1, dtype=jump_indices.dtype),
        jump_indices,
    ])

    database[noise]["x_quenched_fps"] = x_quenched_fps
    database[noise]["w_quenched_fps"] = w_quenched_fps
    database[noise]["num_quenched_fps"] = num_quenched_fps
    database[noise]["jump_indices"] = jump_indices

    print(torch.argwhere(torch.diff(num_quenched_fps) < 0).flatten())

In [ ]:
for noise, data in database.items():

    w_fps = data["w_quenched_fps"]
    betas = data["betas"]

    edgelist_from = []
    edgelist_to = []
    edge_weights = []

    # --------------------------------------------------
    # Match every FP to its closest FP at previous beta
    # --------------------------------------------------

    for beta_idx in range(1, len(betas)):

        w_fp = w_fps[beta_idx]
        w_fp_prev = w_fps[beta_idx - 1]

        dist = fisher_distance(
            w_fp[:, None, :],
            w_fp_prev[None, :, :],
            w_fp_prev[None, :, :],
        )

        closest_prev_idx = dist.argmin(dim=1)

        for idx in range(len(w_fp)):

            parent_idx = closest_prev_idx[idx].item()

            # old -> new
            edgelist_from.append(
                (beta_idx - 1, parent_idx)
            )

            edgelist_to.append(
                (beta_idx, idx)
            )

            edge_weights.append(
                dist[idx, parent_idx].item()
            )

    # --------------------------------------------------
    # Nodes
    # --------------------------------------------------

    fp_nodes = []

    for beta_idx in range(len(w_fps)):
        for idx in range(len(w_fps[beta_idx])):
            fp_nodes.append((beta_idx, idx))

    # --------------------------------------------------
    # DAG
    # --------------------------------------------------

    G = nx.DiGraph()

    G.add_nodes_from(fp_nodes)

    G.add_weighted_edges_from(
        zip(
            edgelist_from,
            edgelist_to,
            edge_weights,
        )
    )

    branching_nodes = [
    node for node in G.nodes
    if G.out_degree(node) > 1
    ]

    print(branching_nodes)

### Experiments

In [ ]:
def build_weighted_simplices(
    W: torch.Tensor,
    n: int,
):
    """
    Build all simplices from edges (r=2) up to simplex size r=n.

    W : [K, N]
        K clusters, N points.

    Returns
    -------
    out[r]["simplices"] : [M_r, r]
    out[r]["strengths"] : [M_r]

    strength(a1,...,ar) = max_i min_j W[aj, i]
    """

    if W.ndim != 2:
        raise ValueError("W must have shape [K, N]")

    K = W.shape[0]
    vertices = torch.arange(K, device=W.device)

    out = {}

    for r in range(2, n + 1):

        # Not enough clusters to form an r-simplex
        if r > K:
            break

        simplices = torch.combinations(vertices, r=r)

        strengths = (
            W[simplices]
            .amin(dim=1)    # min over clusters
            .amax(dim=1)    # max over points
        )

        out[r] = {
            "simplices": simplices,
            "strengths": strengths,
        }

    return out
from scipy.sparse import coo_matrix, csr_matrix
import math
max_order = 3
strength_multiplier = torch.logspace(-6,0,100)
frac_num_simplices = torch.zeros(len(betas), len(strength_multiplier), max_order-1)
edge_matrices = {}
for beta_idx in tqdm(range(len(betas))):
    w_fp = w_quenched_fps[beta_idx]
    found_simplices = build_weighted_simplices(w_fp, max_order)
    for order in range(2, max_order+1):
        if order not in found_simplices:
            break
        simplices = found_simplices[order]["simplices"]
        strengths = found_simplices[order]["strengths"]
        frac_num_simplices[beta_idx, :, order-2] = (strengths[:,None] >= strength_multiplier[None,:]).sum(dim=0)/ math.comb(w_fp.shape[0], order)
        if order == 2:
            edge_matrix = coo_matrix((strengths.cpu().numpy(), (simplices[:,0].cpu().numpy(), simplices[:,1].cpu().numpy())), shape=(w_fp.shape[0], w_fp.shape[0]))
            edge_matrix = (edge_matrix + edge_matrix.T)
            edge_matrices[beta_idx] = edge_matrix.tocsr()


tot_num_triples = torch.zeros(len(betas), len(strength_multiplier), dtype=torch.int32)
tot_num_edges = torch.zeros(len(betas), len(strength_multiplier), dtype=torch.int32)
for beta_idx in range(len(betas)):
    w_fp = w_quenched_fps[beta_idx]

    if len(w_fp) < 2:
        continue

    pairs = torch.combinations(torch.arange(w_fp.shape[0]), r=2)

    pair_strength = torch.minimum(
        w_fp[pairs[:, 0]],
        w_fp[pairs[:, 1]],
    ).max(dim=1).values

    num_edges = (pair_strength[None,...] >= strength_multiplier[:, None]/pair_strength.shape[0]).sum(dim=1)
    tot_num_edges[beta_idx, :] = num_edges

    if len(w_fp) < 3:
        continue

    triples = torch.combinations(torch.arange(w_fp.shape[0]), r=3)

    triple_strength = torch.minimum(
        torch.minimum(
            w_fp[triples[:, 0]],
            w_fp[triples[:, 1]],
        ),
        w_fp[triples[:, 2]],
    ).max(dim=1).values
    
    num_triples = (triple_strength[None,...] >= strength_multiplier[:, None]/triple_strength.shape[0]).sum(dim=1)
    tot_num_triples[beta_idx, :] = num_triples

In [ ]:
def renormalization_step(x, betas = None, device = None, num_steps = 1000):
    from scipy.spatial import KDTree
    anchors  =  torch.as_tensor(x, dtype=torch.float32, device='cpu')
    if device is None:
        device = x.device
    if betas is None:
        anchors_tree = KDTree(anchors)
        lengthscale_min = 0.1*anchors_tree.query(anchors, k=2)[0][:, 1].mean()
        lengthscale_max = 10.0*torch.std(anchors, dim=0).mean() 
        betas = torch.linspace(1/(lengthscale_max**2), 1/(lengthscale_min**2), 1000).to(device)
    else:
        betas = torch.as_tensor(betas, dtype=torch.float32, device=device)
    anchors = anchors.to(device)
    x = torch.clone(anchors).to(device)
    biases = -0.5*(anchors.pow(2)).sum(dim=-1)

    x, weights = mhn_utils.deterministic_dynamics(anchors, biases, betas, x, num_iterations=num_steps, return_probs = True)
    x = x.squeeze(1)
    weights = weights.squeeze(1)
    anchors = anchors.detach().cpu()
    x= x.detach().cpu()
    weights = weights.detach().cpu()
    biases = biases.detach().cpu()
    betas = betas.detach().cpu()

    full_entropies = mhn_utils.get_entropy(weights)

    from math import sqrt, log
    x_fps = []
    w_fps = []
    entropies = []
    entropy_gaps = []
    for beta_idx in range(x.shape[0]):
        x_fp = mhn_utils.group_by_distance(x[beta_idx], eps=1/sqrt(betas[beta_idx].cpu().item()))[0]
        logits = x_fp @ anchors.T
        logits += biases
        logits *= betas[beta_idx].view(-1, 1)
        w_fp = torch.softmax(logits, dim=-1)
        entropies.append(mhn_utils.get_entropy(w_fp))
        entropy_gaps.append(entropies[-1].max() - entropies[-1].min())
        w_fps.append(w_fp)
        x_fps.append(x_fp)
    entropy_gaps = torch.as_tensor(entropy_gaps)
    beta_opt_idx = torch.argmax(entropy_gaps).item()
    beta_opt_idx = torch.argmax(full_entropies.max(dim=1).values - full_entropies.min(dim=1).values).item()
    #beta_opt_idx = torch.argmax(entropies.max(dim=1).values - entropies.min(dim=1).values).item()
    return x_fps[beta_opt_idx], w_fps[beta_opt_idx], beta_opt_idx, betas, entropy_gaps

In [ ]:
test_patterns = []
num_per_center = 10
sigma_test = 0.1
for idx in range(20):
    center = torch.randn(2)
    test_patterns.append(center + torch.randn((num_per_center, 2)) * sigma_test)
test_patterns = torch.cat(test_patterns, dim=0)

In [ ]:
patterns_renorm = torch.clone(patterns)
patterns_renorm -= patterns_renorm.mean(dim=0, keepdim=True)
patterns_renorm /= torch.linalg.norm(patterns_renorm, dim=1, keepdim=True).mean()
patterns_renorm_og = torch.clone(patterns_renorm)
betas_renorm = None
for step in range(10): 
    x_renorm, w_renorm, beta_opt_idx, betas_renorm, entropy_gaps = renormalization_step(patterns_renorm, betas=betas_renorm, num_steps=10000, device='cuda')
    print(f"Step {step}. Beta optimal index: {beta_opt_idx}. Beta value: {betas_renorm[beta_opt_idx]:.4f}")
    
    patterns_renorm = x_renorm
    
    plt.scatter(patterns_renorm_og[:,0], patterns_renorm_og[:,1], alpha=0.5)
    plt.scatter(patterns_renorm[:,0], patterns_renorm[:,1])
    plt.show()
    #plt.plot(betas_renorm, entropy_gaps, marker='o')
    #plt.show()
    print(patterns_renorm.shape[0])
    if beta_opt_idx == 0:
        break

In [ ]:
patterns_renorm = torch.clone(patterns)
for step in range(20):
    patterns_renorm, weights_renorm, beta_opt_idx,   _ = renormalization_step(patterns_renorm, betas = betas, device='cuda', num_steps = 20000)
    
    latents_renorm = (patterns_renorm*rms + shift).view(-1,C_latents, H_latents, W_latents)
    images_renorm = ((1+ae_model.decode(latents_renorm.to(device)).sample.cpu().clamp(-1,1))/2).cpu()
    fig, axs = plt.subplots(ncols=images_renorm.shape[0], figsize=(4*images_renorm.shape[0],4))
    if images_renorm.shape[0] == 1:
        axs = [axs]
    for idx in range(images_renorm.shape[0]):
        axs[idx].imshow(images_renorm[idx].permute(1,2,0).clip(0,1))
    fig.suptitle(f"Step {step}, Beta {betas[beta_opt_idx]:.2f}, Index {beta_opt_idx} out of {len(betas)}")
    plt.show()
    if patterns_renorm.shape[0] == 1:
        break